# Bundle Migration: Generate & Deploy (Modular)

## Overview

Generate Databricks Asset Bundle and deploy to target workspace using centralized configuration.

**Prerequisites:**
- ✅ Bundle_01 completed (dashboards exported and transformed)
- ✅ Config file updated with target workspace details

**What this notebook does:**
1. Loads transformed dashboards and permissions
2. Generates complete bundle structure
3. Validates bundle configuration
4. Deploys to target workspace via CLI
5. Verifies deployment

---

## Cell 0: Install Dependencies

In [ ]:
%pip install -U databricks-sdk pandas pyyaml databricks-cli --quiet
dbutils.library.restartPython()

## Cell 1: Import Helpers & Load Config

In [ ]:
import sys
import os
from pathlib import Path

# Dynamically locate helpers directory - works in both local and workspace contexts
try:
    # In Databricks workspace/job context
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    # Get parent directory of Bundle folder
    bundle_parent = os.path.dirname(os.path.dirname(notebook_path))
    helpers_path = os.path.join('/Workspace', bundle_parent.lstrip('/'), 'helpers')
except:
    # Fallback for local execution
    helpers_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'helpers'))

helpers_path = os.path.normpath(helpers_path)

# Add helpers to path if not already there
if helpers_path not in sys.path:
    sys.path.insert(0, helpers_path)

print(f"📂 Helpers path: {helpers_path}")

from helpers import (
    load_config,
    get_path,
    write_volume_file,
    ensure_directory_exists,
    create_workspace_client
)

print("✅ Helper modules imported")

## Cell 2: Load Configuration

In [ ]:
print("="*80)
print("LOADING CONFIGURATION")
print("="*80)

# Load configuration
config = load_config('../config/config.yaml')

print("\n✅ Configuration loaded\n")
print(f"   Target workspace: {config['target']['workspace_url']}")
print(f"   Bundle name: {config['bundle']['name']}")
print(f"   Parent path: {config['paths']['target_parent_path']}")
print(f"   Warehouse: {config['warehouse'].get('warehouse_name', config['warehouse'].get('warehouse_id', 'Not set'))}")

## Cell 3: Load Transformed Dashboards & Permissions

In [ ]:
print("\n" + "="*80)
print("LOADING DASHBOARDS & PERMISSIONS")
print("="*80)

# Get paths
transformed_path = get_path('transformed')
export_path = get_path('exported')

# List transformed dashboard files
print(f"\n📂 Loading transformed dashboards from: {transformed_path}")
dashboard_files = list_volume_files(transformed_path, '*.lvdash.json')

if not dashboard_files:
    print("   ❌ No transformed dashboards found!")
    print("   💡 Run Bundle_01 first to export and transform dashboards")
    raise Exception("No dashboards to deploy")

print(f"   ✅ Found {len(dashboard_files)} dashboard(s)")

# Load permissions
print(f"\n🔐 Loading permissions from: {export_path}")
permissions_map = load_permissions_from_volume(export_path)
print(f"   ✅ Loaded permissions for {len(permissions_map)} dashboard(s)")

# Build dashboard list with metadata
dashboards = []
for file_path in dashboard_files:
    filename = Path(file_path).name
    
    # Extract dashboard ID and name from filename
    # Format: dashboard_{id}_{name}.lvdash.json
    parts = filename.replace('.lvdash.json', '').split('_', 2)
    if len(parts) >= 3:
        dashboard_id = parts[1]
        name = parts[2].replace('_', ' ')
    else:
        dashboard_id = filename
        name = filename
    
    dashboards.append({
        'id': dashboard_id,
        'name': name,
        'json_path': file_path
    })

# Display summary
df = pd.DataFrame([{'Dashboard': d['name'], 'ID': d['id']} for d in dashboards])
display(df)

## Cell 4: Generate Bundle Structure

In [ ]:
print("\n" + "="*80)
print("GENERATING BUNDLE STRUCTURE")
print("="*80)

bundle_output = get_path('bundles')

print(f"\n📦 Generating bundle: {config['bundle']['name']}")
print(f"   Output path: {bundle_output}")

try:
    bundle_path = generate_bundle_structure(
        bundle_name=config['bundle']['name'],
        target_workspace_url=config['target']['workspace_url'],
        transformed_dashboards=dashboards,
        permissions_map=permissions_map,
        bundle_output_path=bundle_output,
        warehouse_id=config['warehouse'].get('warehouse_id'),
        warehouse_name=config['warehouse'].get('warehouse_name'),
        parent_path=config['paths']['target_parent_path'],
        embed_credentials=config['bundle']['embed_credentials']
    )
    
    print(f"\n✅ Bundle generated at: {bundle_path}")
    print("\n📋 Bundle structure:")
    print("   ├── databricks.yml")
    print("   ├── resources/")
    print("   │   └── dashboards.yml")
    print("   └── src/")
    print("       └── dashboards/")
    print(f"           └── {len(dashboards)} .lvdash.json file(s)")
    
except Exception as e:
    print(f"\n❌ Failed to generate bundle: {e}")
    raise

## Cell 5: Validate Bundle

In [ ]:
print("\n" + "="*80)
print("VALIDATING BUNDLE")
print("="*80)

print("\n🔍 Running: databricks bundle validate")

# Convert volume path to local path for CLI
# Bundle CLI needs local filesystem path
local_bundle_path = bundle_path.replace('/Volumes/', '/dbfs/Volumes/')

result = validate_bundle(local_bundle_path)

if result['success']:
    print("\n✅ Bundle validation passed")
    if result.get('stdout'):
        print(f"\n{result['stdout']}")
else:
    print("\n❌ Bundle validation failed")
    if result.get('stderr'):
        print(f"\nError: {result['stderr']}")
    if result.get('error'):
        print(f"\nError: {result['error']}")
    raise Exception("Bundle validation failed")

## Cell 6: Deploy Bundle

In [ ]:
print("\n" + "="*80)
print("DEPLOYING BUNDLE")
print("="*80)

print(f"\n🚀 Deploying to: {config['target']['workspace_url']}")
print("   This may take a few minutes...")

deploy_result = deploy_bundle(local_bundle_path)

if deploy_result['success']:
    print("\n✅ Bundle deployed successfully!")
    
    if deploy_result.get('stdout'):
        print(f"\n{deploy_result['stdout']}")
    
    print("\n" + "="*80)
    print("DEPLOYMENT SUMMARY")
    print("="*80)
    print(f"\n📊 Dashboards deployed: {len(dashboards)}")
    print(f"📁 Location: {config['paths']['target_parent_path']}")
    print(f"🔗 Workspace: {config['target']['workspace_url']}")
    
    print("\n✅ Next steps:")
    print("   1. Navigate to target workspace")
    print(f"   2. Go to {config['paths']['target_parent_path']}")
    print("   3. Verify dashboards load correctly")
    print("   4. Test data with new catalog/schema")
    print("   5. Verify permissions applied")
    
else:
    print("\n❌ Bundle deployment failed")
    
    if deploy_result.get('stderr'):
        print(f"\nError output:\n{deploy_result['stderr']}")
    if deploy_result.get('error'):
        print(f"\nError: {deploy_result['error']}")
    
    print("\n💡 Troubleshooting:")
    print("   - Check target workspace credentials")
    print("   - Verify warehouse exists and is accessible")
    print("   - Check parent folder permissions")
    print("   - Review bundle validation output above")
    
    raise Exception("Bundle deployment failed")

## Cell 7: Verify Deployment

In [ ]:
print("\n" + "="*80)
print("VERIFYING DEPLOYMENT")
print("="*80)

print("\n🔍 Connecting to target workspace...")
target_client = create_workspace_client('target')

print("\n📋 Listing deployed dashboards...")
deployed_dashboards = []

for dash in target_client.lakeview.list():
    if dash.parent_path and dash.parent_path.startswith(config['paths']['target_parent_path']):
        deployed_dashboards.append({
            'Name': dash.display_name,
            'ID': dash.dashboard_id,
            'Path': dash.parent_path
        })

if deployed_dashboards:
    print(f"\n✅ Found {len(deployed_dashboards)} deployed dashboard(s)\n")
    df = pd.DataFrame(deployed_dashboards)
    display(df)
    
    print("\n🎉 Migration complete!")
    print("\n📝 Verification checklist:")
    print("   □ Dashboards visible in target workspace")
    print("   □ Data loads correctly (new catalog/schema)")
    print("   □ Visualizations render properly")
    print("   □ Permissions applied correctly")
    print("   □ Warehouse connection works")
else:
    print("\n⚠️  No dashboards found at expected location")
    print(f"   Expected path: {config['paths']['target_parent_path']}")
    print("\n💡 Check deployment logs above for errors")